# Barrido de BioBERT sobre una partición sin fuga

Vuelve a correr el barrido de 24 configuraciones del servidor, pero sobre una
partición que **no** deja la misma ventana de texto en entrenamiento y en prueba.

Sobre los archivos originales, el 73.9 % de los ejemplos de prueba tenía su ventana
ya vista en entrenamiento, y 61 de los 64 artículos de prueba estaban también en
entrenamiento. Por eso el macro-F1 reportado —0.9024 en dev, 0.8721 en test— mide
sobre todo memorización.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno → GPU*.

El barrido es reanudable. Si la sesión se cae, vuelve a correr la última celda y
retoma donde iba.

## 1. Comprobar que hay GPU

In [ ]:
# En subproceso a proposito: nada de torch, transformers ni numpy dentro del
# kernel del cuaderno. Todo lo pesado ya corre por `!python ...`, asi que
# manteniendo el kernel limpio no hay que reiniciar el entorno despues del pip
# ni existe el riesgo de que un modulo viejo en memoria contradiga al que
# quedo en disco.
import subprocess, sys

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

r = subprocess.run([sys.executable, '-c',
    "import torch;"
    "print('torch', torch.__version__);"
    "print('CUDA disponible:', torch.cuda.is_available());"
    "raise SystemExit(0 if torch.cuda.is_available() else 1)"])
if r.returncode:
    raise SystemExit('Sin GPU. Entorno de ejecucion -> Cambiar tipo de '
                     'entorno de ejecucion -> GPU.')

## 2. Fijar las versiones

Las del servidor. Importa: el script de entrenamiento usa `evaluation_strategy=`,
que **desapareció en transformers 4.46** (ahora se llama `eval_strategy`), y su
`WeightedTrainer.compute_loss` tiene la firma vieja. Con la versión de Colab de
hoy truena en la primera corrida.

`pip` va a escupir una lista larga de conflictos: jax, cupy, opencv, rasterio y
compañía quieren numpy 2, y aquí se baja a 1.x. **Son paquetes que este cuaderno
nunca importa.** El aviso es de pip, que revisa el entorno entero y no solo lo
que se acaba de instalar; por eso también aparecen conflictos que ya venían en la
imagen de Colab (requests, cuda-toolkit, torchvision) y que nada tienen que ver
con esto. Lo que sí hay que leer es la comprobación de abajo, que dice versión
por versión si quedó lo que se pidió.

**No hace falta reiniciar el entorno.** Las comprobaciones corren en un
subproceso, que lee lo que hay en disco y no lo que quedó en memoria.

In [ ]:
# pandas va fijado junto con numpy y no por gusto: el pandas 3 de la imagen de
# Colab exige numpy 2, y aqui numpy tiene que ser 1.x porque datasets 2.20 usa
# np.float_, que numpy 2 elimino. Sin fijarlo quedan dos paquetes que se
# contradicen y `import datasets` puede tronar a media corrida. La 2.2.3 es
# ademas la que pide google-colab, asi que no rompe el montaje de Drive.
!pip install -q --no-warn-conflicts \
    "transformers==4.44.2" "datasets==2.20.0" "accelerate==0.33.0" \
    "numpy<2" "pandas==2.2.3" scikit-learn

# Comprobar de verdad, no confiar en que el pip salio bien. Sin esto, una
# incompatibilidad de versiones hace fallar las 24 corridas una por una: el
# barrido las anota como error y sigue, y media hora despues hay 24 archivos
# .error y ni un solo resultado.
#
# Dos cosas que esta comprobacion NO hace, y cada una corrige un tropiezo real:
#
# - No importa nada en el kernel del cuaderno. Si numpy o pandas ya estaban
#   cargados en memoria cuando corrio el pip, `import numpy` devuelve el viejo
#   y la comprobacion miente en cualquiera de las dos direcciones.
# - No construye un TrainingArguments. Instanciarlo dispara la busqueda de
#   integraciones (wandb, mlflow, comet, neptune, clearml...) y el arranque de
#   CUDA; en un entorno recien reinstalado eso se va varios minutos haciendo
#   stat() sobre sys.path y parece colgado. La firma de la clase responde
#   exactamente la misma pregunta y es instantanea.
import subprocess, sys

COMPROBAR = r"""
import importlib, inspect

for mod, quiero in (('transformers', '4.44.2'), ('datasets', '2.20.0'),
                    ('accelerate', '0.33.0')):
    try:
        m = importlib.import_module(mod)
    except ImportError as e:
        raise SystemExit('No quedo instalado %s (%s).' % (mod, e))
    hay = getattr(m, '__version__', '?')
    print('%-14s %-10s %s' % (mod, hay, 'ok' if hay == quiero else 'ESPERABA ' + quiero))
    if hay != quiero:
        raise SystemExit('Version equivocada de %s.' % mod)

import numpy
print('%-14s %-10s %s' % ('numpy', numpy.__version__,
                          'ok' if numpy.__version__.startswith('1.') else 'ESPERABA 1.x'))
if not numpy.__version__.startswith('1.'):
    raise SystemExit('numpy 2.x rompe datasets 2.20 (np.float_).')

import pandas
print('%-14s %-10s %s' % ('pandas', pandas.__version__,
                          'ok' if pandas.__version__.startswith('2.') else 'ESPERABA 2.x'))
if not pandas.__version__.startswith('2.'):
    raise SystemExit('pandas 3.x exige numpy 2 y aqui numpy es 1.x.')

# La prueba que de verdad importa: que el script del servidor se pueda usar
# con esta version. TrainingArguments perdio evaluation_strategy en la 4.46.
from transformers import TrainingArguments
if 'evaluation_strategy' not in inspect.signature(TrainingArguments.__init__).parameters:
    raise SystemExit('Esta transformers ya no acepta evaluation_strategy.')
print('\nevaluation_strategy sigue existiendo: el script del servidor corre.')
"""

if subprocess.run([sys.executable, '-c', COMPROBAR]).returncode:
    raise SystemExit('El entorno no quedo como se pidio. Mira el mensaje de '
                     'arriba y vuelve a correr esta celda.')

## 3. Traer los archivos

Sube a tu Drive una carpeta `pseudomonas-trn/entrada` con **cinco archivos**:

- `entity_marked_train.jsonl`, `entity_marked_dev.jsonl`, `entity_marked_test.jsonl`
- `bio_bert_re_finetune.py`
- `particionar.py` y `barrido.py` (de `etapa2/` en el repositorio)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
RAIZ    = '/content/drive/MyDrive/pseudomonas-trn'
ENTRADA = os.path.join(RAIZ, 'entrada')
os.makedirs('/content/trabajo', exist_ok=True)

falta = [f for f in ('entity_marked_train.jsonl', 'entity_marked_dev.jsonl',
                     'entity_marked_test.jsonl', 'bio_bert_re_finetune.py',
                     'particionar.py', 'barrido.py')
         if not os.path.exists(os.path.join(ENTRADA, f))]
if falta:
    raise SystemExit('Faltan en %s: %s' % (ENTRADA, falta))

for f in os.listdir(ENTRADA):
    shutil.copy(os.path.join(ENTRADA, f), '/content/trabajo/')
os.chdir('/content/trabajo')
print('Copiados:', sorted(os.listdir('.')))

## 4. Reparticionar y verificar

Agrupa por **artículo + ventana de texto** y comprueba que no quedó fuga. Si
queda, sale con error y no hay que gastar GPU.

Sale en 1242 / 163 / 157, casi igual que la partición original (1249 / 156 / 157)
y con las mismas proporciones de clase. La comparación es manzana con manzana:
lo único que cambia es la fuga.

In [ ]:
!python particionar.py --por pmid --salida limpia_por_pmid --intentos 120

## 4b. Prueba de humo: una corrida diminuta

Antes de gastar horas de GPU, correr el script de verdad sobre 40 ejemplos, una
época y ventanas de 128. Tarda un par de minutos y ejerce el camino completo que
el barrido va a repetir 24 veces: `Trainer`, la firma vieja de `compute_loss`,
los pesos por clase, `save_safetensors=False` y la evaluación. Una versión
incompatible que la comprobación del paso 2 no alcanza a ver —de `torch` o de
`accelerate`, por ejemplo— sale aquí en dos minutos, no en media hora de
corridas fallidas.

De paso deja BioBERT en la caché, así que la primera corrida del barrido ya no
paga la descarga.

Si truena con un error de `accelerate` o de `torch`: el par transformers 4.44 /
accelerate 0.33 es de 2024 y Colab actualiza torch cada tanto, así que la salida
es fijar también torch con `!pip install -q "torch==2.4.1"` y volver a la celda
de comprobación. Es una descarga de más de un giga, y por eso no va puesta de
entrada.

In [ ]:
# 40 ejemplos tomados a saltos, no los primeros 40: el corpus viene agrupado
# por articulo, asi que los primeros suelen ser del mismo PMID y de la misma
# clase, y una prueba de humo con una sola clase no ejerce los pesos por clase.
import os, shutil, subprocess, sys

HUMO = '/content/_humo'
shutil.rmtree(HUMO, ignore_errors=True)
os.makedirs(HUMO)

for nombre, cuantos in (('train', 40), ('dev', 16), ('test', 16)):
    with open('limpia_por_pmid/entity_marked_%s.jsonl' % nombre,
              encoding='utf-8') as f:
        lineas = f.readlines()
    paso = max(1, len(lineas) // cuantos)
    with open(os.path.join(HUMO, 'entity_marked_%s.jsonl' % nombre), 'w',
              encoding='utf-8') as f:
        f.writelines(lineas[::paso][:cuantos])

shutil.copy('limpia_por_pmid/label_mapping.json', HUMO)

cmd = [sys.executable, 'bio_bert_re_finetune.py',
       '--train_jsonl', HUMO + '/entity_marked_train.jsonl',
       '--dev_jsonl',   HUMO + '/entity_marked_dev.jsonl',
       '--test_jsonl',  HUMO + '/entity_marked_test.jsonl',
       '--labels_json', HUMO + '/label_mapping.json',
       '--out_dir',     HUMO + '/salida',
       '--model_name',  'dmis-lab/biobert-base-cased-v1.1',
       '--batch_size', '8', '--epochs', '1', '--max_length', '128',
       '--lr', '3e-5', '--warmup_ratio', '0.1', '--seed', '42',
       '--use_class_weights']
codigo = subprocess.run(cmd).returncode

# Los pesos se borran gane o pierda: son ~433 MB que no sirven para nada, y el
# barrido necesita el disco local para sus propios checkpoints.
shutil.rmtree(HUMO, ignore_errors=True)

if codigo:
    raise SystemExit('La prueba de humo fallo. No lances el barrido: las 24 '
                     'corridas fallarian igual, una por una.')
print('\nPrueba de humo ok: el barrido puede correr.')

## 5. El barrido

Las 24 configuraciones exactas de `run_sweep_biobert.sh`, con su mismo orden y su
misma numeración: aquí `run_22` es también `lr3e-5 ep8 bs16 wu0.1`, la que el
servidor reportó como mejor. Así se compara corrida contra corrida, no solo el
mejor de cada barrido.

Cada una guarda su resultado en Drive apenas termina, así que una desconexión
cuesta una corrida, no el barrido.

Los checkpoints van a `/content` (disco local), **nunca a Drive**: son ~433 MB por
época y mandarlos por red tardaría más que entrenar.

Si la sesión se cae, vuelve a correr esta misma celda.

In [ ]:
!python barrido.py \
    --datos      limpia_por_pmid \
    --script     bio_bert_re_finetune.py \
    --trabajo    /content/runs \
    --resultados /content/drive/MyDrive/pseudomonas-trn/barrido_por_pmid

## 6. La tabla

Se puede correr en cualquier momento, incluso a media sesión, para ver lo que va
habiendo. Escribe `resumen.csv` junto a los resultados.

In [ ]:
!python barrido.py --solo_resumen --datos limpia_por_pmid \
    --resultados /content/drive/MyDrive/pseudomonas-trn/barrido_por_pmid

## 7. Opcional: la partición por par, con validación cruzada

Responde una pregunta distinta: **¿generaliza a relaciones que nunca vio?**

No se puede hacer al mismo tiempo que la anterior. Exigir que no se comparta ni
artículo ni par colapsa el corpus en una sola componente con el 89 % de los
ejemplos: es imposible partirlo. Hay que elegir, y por eso van por separado.

Con 5 pliegues son 5 corridas por configuración. Conviene correrlo **solo con la
mejor configuración** que haya salido del paso 5, no con las 24.

In [ ]:
!python particionar.py --por par --salida limpia_por_par --folds 5 --intentos 60

In [ ]:
# Un barrido por pliegue, cada uno con su propia carpeta de resultados: si
# compartieran una, la huella de los datos haria que el segundo se negara a
# correr, que es exactamente lo que debe pasar.
#
# --solo 22 corre UNA configuracion, la ganadora del paso 5. Las 24 en cada
# uno de los 5 pliegues serian 120 corridas y no aportan nada: los
# hiperparametros ya se eligieron.
GANADORA = 22   # ajusta con lo que diga resumen.csv del paso 6

for i in range(5):
    print('\n' + '=' * 60 + '\npliegue %d\n' % i + '=' * 60)
    !python barrido.py \
        --datos      limpia_por_par/fold_{i} \
        --script     bio_bert_re_finetune.py \
        --solo       {GANADORA} \
        --trabajo    /content/runs \
        --resultados /content/drive/MyDrive/pseudomonas-trn/barrido_por_par/fold_{i}

In [ ]:
# El promedio de los cinco pliegues, con su dispersion. La dispersion es la
# mitad util del resultado: dice si la diferencia contra el barrido por
# articulo es senal o es ruido de particion.
import glob, json, statistics

f1s = []
for d in sorted(glob.glob('/content/drive/MyDrive/pseudomonas-trn/barrido_por_par/fold_*')):
    for r in glob.glob(d + '/run_*.json'):
        m = json.load(open(r))
        if m.get('test_macro_f1') is not None:
            f1s.append((d.split('/')[-1], m['test_macro_f1']))

for n, v in sorted(f1s):
    print('%-10s test macro-F1 %.4f' % (n, v))

if len(f1s) >= 2:
    v = [x[1] for x in f1s]
    print('\n%d pliegues: %.4f +- %.4f  (min %.4f, max %.4f)'
          % (len(v), statistics.mean(v), statistics.stdev(v), min(v), max(v)))
elif f1s:
    print('\nSolo %d pliegue: falta correr los demas.' % len(f1s))
else:
    print('\nNo hay resultados todavia.')

## 8. Bajar la mejor corrida

Los pesos se borran al terminar cada configuración para no llenar el disco. Para
quedarte con los de la ganadora, vuelve a correrla sola con `--conservar`
usando los valores que haya dado la tabla.

In [ ]:
# Ajusta los cuatro valores con lo que diga resumen.csv.
# Los demas son los fijos del .sh del servidor: no los cambies o deja de ser
# el mismo experimento.
LR, EPOCHS, BATCH, WARMUP = '3e-5', 8, 16, 0.1

!python bio_bert_re_finetune.py \
    --train_jsonl limpia_por_pmid/entity_marked_train.jsonl \
    --dev_jsonl   limpia_por_pmid/entity_marked_dev.jsonl \
    --test_jsonl  limpia_por_pmid/entity_marked_test.jsonl \
    --labels_json limpia_por_pmid/label_mapping.json \
    --out_dir     /content/mejor \
    --model_name  dmis-lab/biobert-base-cased-v1.1 \
    --batch_size {BATCH} --epochs {EPOCHS} --lr {LR} --warmup_ratio {WARMUP} \
    --weight_decay 0.01 --max_length 512 --seed 42 \
    --use_class_weights --early_stopping --early_stopping_patience 2

# Solo los pesos y el tokenizador; los optimizer.pt son los que hincharon el
# arbol del servidor a 75 GB y no sirven para inferir.
!mkdir -p /content/drive/MyDrive/pseudomonas-trn/mejor_sin_fuga
!cp /content/mejor/pytorch_model.bin /content/mejor/config.json \
    /content/mejor/tokenizer_config.json /content/mejor/special_tokens_map.json \
    /content/mejor/vocab.txt \
    /content/drive/MyDrive/pseudomonas-trn/mejor_sin_fuga/
!ls -la /content/drive/MyDrive/pseudomonas-trn/mejor_sin_fuga/